# Power Outage Forecasting — LSTM Models

This notebook builds two LSTM models to forecast power outages across 83 counties for 24h and 48h horizons.

We compare against four baseline models from `baseline_yuwenz.ipynb`:
- **Persistence**: predict tomorrow = today
- **Historical Median**: predict using the median outage at the same hour
- **Hurdle**: two-stage model (logistic + Poisson)
- **ZIP**: Zero-Inflated Poisson

Our two LSTM models:
- **LSTM-Simple**: uses top 10 weather features most correlated with outages
- **LSTM-Full**: uses all 77 valid weather features, letting the model learn on its own

The motivation is that LSTM can capture temporal dependencies in the outage time series, and adding weather features should help the model understand what external conditions drive outages.

## 1. Setup & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

PROJECT_PATH = '/content/drive/MyDrive/MLPS'
os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DATA_DIR    = "data/"
RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH    = os.path.join(DATA_DIR, "train.nc")
TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h_demo.nc")
TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h_demo.nc")

VALIDATION_SPLIT = 0.2
SEQ_LEN          = 24
BATCH_SIZE       = 64
EPOCHS           = 20
LR               = 1e-3
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

## 2. Data Loading & Cleaning

We apply the same cleaning pipeline as in `baseline_yuwenz.ipynb` to keep things consistent:
1. **IQR outlier clipping** — weather values outside 1.5×IQR are replaced with NaN
2. **Linear interpolation** — fills the NaN gaps along the time dimension

The outage variable (`out`) is kept as-is. We then do a temporal 80/20 train/val split.

In [ ]:
ds_train    = xr.open_dataset(TRAIN_PATH)
ds_test_24h = xr.open_dataset(TEST_24H_PATH)
ds_test_48h = xr.open_dataset(TEST_48H_PATH)

train_timestamps = pd.to_datetime(ds_train.timestamp.values)
locations        = [str(x) for x in ds_train.location.values]
weather_features = list(ds_train.feature.values)

print(f"Training Period  : {train_timestamps.min()} to {train_timestamps.max()}")
print(f"Number of Locations: {len(locations)}")
print(f"Weather Features ({len(weather_features)}): {weather_features}")

In [ ]:
# IQR outlier clipping
weather = ds_train.weather.copy()
Q1 = weather.quantile(0.25, dim="timestamp")
Q3 = weather.quantile(0.75, dim="timestamp")
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
weather_clean = weather.where((weather >= lower) & (weather <= upper))

# Linear interpolation
weather_filled = weather_clean.interpolate_na(dim="timestamp", method="linear")

# Rebuild clean dataset
ds_train_clean = xr.Dataset(
    {"weather": weather_filled, "tracked": ds_train.tracked, "out": ds_train.out},
    coords={"location": ds_train.location, "timestamp": ds_train.timestamp, "feature": ds_train.feature}
)

# Train/val split
all_timestamps = pd.to_datetime(ds_train_clean.timestamp.values)
T = ds_train_clean.dims["timestamp"]
split_idx = int((1 - VALIDATION_SPLIT) * T)

train_data = ds_train_clean.isel(timestamp=slice(0, split_idx))
val_data   = ds_train_clean.isel(timestamp=slice(split_idx, None))
train_ts   = all_timestamps[:split_idx]
val_ts     = all_timestamps[split_idx:]

print(f"Total timestamps : {T}")
print(f"Train            : {len(train_ts)} steps  ({train_ts[0]} → {train_ts[-1]})")
print(f"Val              : {len(val_ts)} steps  ({val_ts[0]} → {val_ts[-1]})")

## 3. Feature Engineering

The baseline models only use `lag_1`, `lag_24`, `hour`, and `dayofweek`. Here we do a more careful feature selection for the weather inputs.

**Step 1 — Remove all-zero features**: Some weather columns are entirely zero across the dataset. These carry no information and should be dropped.

**Step 2 — Correlation filter**: We compute the absolute correlation between each remaining weather feature and the outage count. Features with NaN correlation (constant or near-constant columns) are also removed.

**Step 3 — Define two feature sets**:
- `LSTM-Simple`: top 10 features by correlation — handpicked, interpretable
- `LSTM-Full`: all 77 valid features — let the model figure out what matters

In addition to weather, both models use:
- `lag_1`, `lag_24` — recent outage history
- `hour_sin/cos`, `dow_sin/cos` — cyclic time encodings (better than raw integers for capturing daily/weekly patterns)

In [ ]:
# Step 1: Remove all-zero features
weather_flat  = weather_filled.values.reshape(-1, weather_filled.shape[-1])
all_zero_mask = np.all(weather_flat == 0, axis=0)
zero_features     = [weather_features[i] for i in range(len(weather_features)) if all_zero_mask[i]]
non_zero_features = [f for f in weather_features if f not in zero_features]

print(f"Excluded all-zero features ({len(zero_features)}): {zero_features}")
print(f"Remaining non-zero features: {len(non_zero_features)}")

# Step 2: Compute correlation with outage target
out_flat   = ds_train.out.values.flatten()
df_weather = pd.DataFrame(weather_flat, columns=weather_features)
df_weather['out'] = out_flat

target_corr = df_weather[non_zero_features + ['out']].corr()['out'].drop('out')

# Remove NaN correlation features
nan_corr_features = target_corr[target_corr.isna()].index.tolist()
valid_features    = [f for f in non_zero_features if f not in nan_corr_features]

print(f"Excluded NaN correlation features ({len(nan_corr_features)}): {nan_corr_features}")
print(f"Final valid weather features: {len(valid_features)}")

# Step 3: Rank and define feature sets
target_corr_valid = target_corr[valid_features].abs().sort_values(ascending=False)
print(f"\nTop 10 weather features by correlation with outage:")
print(target_corr_valid.head(10))

selected_weather_features = target_corr_valid.head(10).index.tolist()  # LSTM-Simple
all_weather_features      = valid_features                              # LSTM-Full

print(f"\nLSTM-Simple weather features ({len(selected_weather_features)}): {selected_weather_features}")
print(f"LSTM-Full   weather features ({len(all_weather_features)}): {len(all_weather_features)} features")

## 4. Build Feature Matrices

We convert the xarray dataset into a 3D numpy array of shape `(T, L, D)` where:
- `T` = number of timestamps
- `L` = number of locations (83 counties)
- `D` = number of features per location-time pair

Features included for each location at each timestep:
- `out` (current outage count)
- `lag_1`, `lag_24` (outage 1h and 24h ago)
- `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos` (cyclic time encodings)
- selected or all weather features

In [ ]:
def build_feature_matrix(ds, ts, weather_feats):
    out_vals = ds.out.transpose("timestamp", "location").values.astype(float)
    T, L = out_vals.shape

    def lag(arr, k):
        shifted = np.full_like(arr, np.nan)
        shifted[k:] = arr[:-k]
        return shifted

    lag1  = lag(out_vals, 1)
    lag24 = lag(out_vals, 24)

    hour_sin = np.sin(2 * np.pi * ts.hour / 24).values[:, None] * np.ones((1, L))
    hour_cos = np.cos(2 * np.pi * ts.hour / 24).values[:, None] * np.ones((1, L))
    dow_sin  = np.sin(2 * np.pi * ts.dayofweek / 7).values[:, None] * np.ones((1, L))
    dow_cos  = np.cos(2 * np.pi * ts.dayofweek / 7).values[:, None] * np.ones((1, L))

    w_vals = ds.weather.sel(feature=weather_feats).transpose(
        "timestamp", "location", "feature"
    ).values.astype(float)

    base = np.stack([out_vals, lag1, lag24, hour_sin, hour_cos, dow_sin, dow_cos], axis=-1)
    X = np.concatenate([base, w_vals], axis=-1)
    return X, out_vals

print("Building feature matrix for LSTM-Simple...")
X_simple, y = build_feature_matrix(ds_train_clean, all_timestamps, selected_weather_features)
print(f"  Shape: {X_simple.shape}  → (T, L, {X_simple.shape[-1]} features)")

print("Building feature matrix for LSTM-Full...")
X_full, _   = build_feature_matrix(ds_train_clean, all_timestamps, all_weather_features)
print(f"  Shape: {X_full.shape}  → (T, L, {X_full.shape[-1]} features)")

## 5. Train/Val Split & Normalization

We normalize features using z-score normalization. Importantly, the scaler is fitted on the **training set only** and then applied to both train and val — this avoids data leakage.

In [ ]:
X_simple_tr, X_simple_va = X_simple[:split_idx], X_simple[split_idx:]
X_full_tr,   X_full_va   = X_full[:split_idx],   X_full[split_idx:]
y_tr,        y_va        = y[:split_idx],         y[split_idx:]

print(f"Train: {X_simple_tr.shape[0]} steps")
print(f"Val  : {X_simple_va.shape[0]} steps")

def fit_scaler(arr):
    flat = arr.reshape(-1, arr.shape[-1])
    mu = np.nanmean(flat, axis=0)
    sd = np.nanstd(flat,  axis=0)
    sd = np.where(sd == 0, 1.0, sd)
    return mu, sd

def apply_scaler(arr, mu, sd):
    return (arr - mu) / sd

mu_simple, sd_simple = fit_scaler(X_simple_tr)
mu_full,   sd_full   = fit_scaler(X_full_tr)

X_simple_tr_n = apply_scaler(np.nan_to_num(X_simple_tr, 0), mu_simple, sd_simple)
X_simple_va_n = apply_scaler(np.nan_to_num(X_simple_va, 0), mu_simple, sd_simple)
X_full_tr_n   = apply_scaler(np.nan_to_num(X_full_tr, 0), mu_full, sd_full)
X_full_va_n   = apply_scaler(np.nan_to_num(X_full_va, 0), mu_full, sd_full)

print("✓ Normalization done")

## 6. Sliding Window Dataset

LSTM models require sequential input. We use a sliding window approach:
- **Input**: the past `SEQ_LEN=24` hours of features for one location
- **Target**: the next `horizon` hours of outage counts

We slide this window across all timestamps and all 83 locations, giving us a large training dataset.

In [ ]:
def build_windows(X_norm, y_raw, seq_len, horizon):
    T, L, D = X_norm.shape
    inputs, targets = [], []
    for loc_idx in range(L):
        x_loc = X_norm[:, loc_idx, :]
        y_loc = y_raw[:, loc_idx]
        for t in range(T - seq_len - horizon + 1):
            inp = x_loc[t : t + seq_len]
            tgt = y_loc[t + seq_len : t + seq_len + horizon]
            if np.isnan(inp).any() or np.isnan(tgt).any():
                continue
            inputs.append(inp)
            targets.append(tgt)
    return np.array(inputs, dtype=np.float32), np.array(targets, dtype=np.float32)

print("Building windows for LSTM-Simple...")
X_win_simple_24, y_win_24 = build_windows(X_simple_tr_n, y_tr, SEQ_LEN, 24)
X_win_simple_48, y_win_48 = build_windows(X_simple_tr_n, y_tr, SEQ_LEN, 48)
print(f"  24h: inputs {X_win_simple_24.shape}  targets {y_win_24.shape}")
print(f"  48h: inputs {X_win_simple_48.shape}  targets {y_win_48.shape}")

print("Building windows for LSTM-Full...")
X_win_full_24, _ = build_windows(X_full_tr_n, y_tr, SEQ_LEN, 24)
X_win_full_48, _ = build_windows(X_full_tr_n, y_tr, SEQ_LEN, 48)
print(f"  24h: inputs {X_win_full_24.shape}  targets {y_win_24.shape}")
print(f"  48h: inputs {X_win_full_48.shape}  targets {y_win_48.shape}")

## 7. Model Architecture

We use a standard LSTM with a two-layer MLP output head. The same architecture is used for both LSTM-Simple and LSTM-Full — the only difference is the input dimension.

Key design choices:
- **2-layer LSTM** with hidden size 64 — enough capacity without being too heavy
- **Dropout 0.2** between layers — regularization to prevent overfitting
- **ReLU + clamp(min=0)** on output — outage counts are always non-negative
- **Gradient clipping** during training — stabilizes LSTM training

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2, horizon=24):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = out[:, -1, :]
        pred   = self.head(last)
        return torch.clamp(pred, min=0.0)

# Sanity check
dummy_simple = torch.zeros(4, SEQ_LEN, X_win_simple_24.shape[-1])
dummy_full   = torch.zeros(4, SEQ_LEN, X_win_full_24.shape[-1])

m = LSTMModel(input_dim=X_win_simple_24.shape[-1], horizon=24)
print(f"LSTM-Simple output shape: {m(dummy_simple).shape}  ✓")
m = LSTMModel(input_dim=X_win_full_24.shape[-1], horizon=24)
print(f"LSTM-Full   output shape: {m(dummy_full).shape}  ✓")

n_params_simple = sum(p.numel() for p in LSTMModel(X_win_simple_24.shape[-1]).parameters())
n_params_full   = sum(p.numel() for p in LSTMModel(X_win_full_24.shape[-1]).parameters())
print(f"\nLSTM-Simple parameters: {n_params_simple:,}")
print(f"LSTM-Full   parameters: {n_params_full:,}")

## 8. Training Function

In [ ]:
def train_model(X_windows, y_windows, input_dim, horizon, epochs=EPOCHS,
                batch_size=BATCH_SIZE, lr=LR, model_name="LSTM"):
    dataset   = TensorDataset(torch.tensor(X_windows), torch.tensor(y_windows))
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model     = LSTMModel(input_dim=input_dim, horizon=horizon).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.MSELoss()
    history   = []
    for epoch in tqdm(range(epochs), desc=f"Training {model_name} {horizon}h"):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= len(dataset)
        history.append(epoch_loss)
        scheduler.step()
    return model, history

## 9. Train LSTM-Simple

LSTM-Simple uses the top 10 weather features selected by correlation analysis. We train separate models for 24h and 48h prediction horizons.

In [ ]:
print("Training LSTM-Simple 24h...")
model_simple_24h, hist_simple_24h = train_model(
    X_win_simple_24, y_win_24, X_win_simple_24.shape[-1], 24, model_name="LSTM-Simple")

print("\nTraining LSTM-Simple 48h...")
model_simple_48h, hist_simple_48h = train_model(
    X_win_simple_48, y_win_48, X_win_simple_48.shape[-1], 48, model_name="LSTM-Simple")

print("\n✓ LSTM-Simple training done!")

# Plot loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("LSTM-Simple Training Loss", fontsize=13, fontweight='bold')
for ax, hist, title in zip(axes, [hist_simple_24h, hist_simple_48h], ['24h', '48h']):
    ax.plot(hist, marker='o', markersize=3, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "loss_simple.png"), dpi=120)
plt.show()

## 10. Train LSTM-Full

LSTM-Full uses all 77 valid weather features. The idea is to give the model more information and let it learn which features are useful through the training process.

In [ ]:
print("Training LSTM-Full 24h...")
model_full_24h, hist_full_24h = train_model(
    X_win_full_24, y_win_24, X_win_full_24.shape[-1], 24, model_name="LSTM-Full")

print("\nTraining LSTM-Full 48h...")
model_full_48h, hist_full_48h = train_model(
    X_win_full_48, y_win_48, X_win_full_48.shape[-1], 48, model_name="LSTM-Full")

print("\n✓ LSTM-Full training done!")

# Plot loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("LSTM-Full Training Loss", fontsize=13, fontweight='bold')
for ax, hist, title in zip(axes, [hist_full_24h, hist_full_48h], ['24h', '48h']):
    ax.plot(hist, marker='o', markersize=3, color='darkorange')
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "loss_full.png"), dpi=120)
plt.show()

## 11. Validation Predictions

In [ ]:
def predict_with_timestamps(model, X_context_norm, horizon, locations, timestamps):
    model.eval()
    context = X_context_norm[-SEQ_LEN:, :, :]
    inp     = torch.tensor(context.transpose(1, 0, 2), dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = model(inp).cpu().numpy()
    rows = []
    for loc_idx, loc in enumerate(locations):
        for h in range(horizon):
            rows.append({"step": h, "location": loc, "pred": float(preds[loc_idx, h])})
    df = pd.DataFrame(rows)
    df['timestamp'] = df['step'].apply(lambda i: timestamps[i] if i < len(timestamps) else None)
    return df[['timestamp', 'location', 'pred']].dropna(subset=['timestamp']).reset_index(drop=True)

print("Generating validation predictions...")
val_simple_24h = predict_with_timestamps(model_simple_24h, X_simple_tr_n, 24, locations, val_ts)
val_simple_48h = predict_with_timestamps(model_simple_48h, X_simple_tr_n, 48, locations, val_ts)
val_full_24h   = predict_with_timestamps(model_full_24h,   X_full_tr_n,   24, locations, val_ts)
val_full_48h   = predict_with_timestamps(model_full_48h,   X_full_tr_n,   48, locations, val_ts)

print(f"Val 24h predictions: {len(val_simple_24h)} rows  (expected {24 * len(locations)})")
print(f"Val 48h predictions: {len(val_simple_48h)} rows  (expected {48 * len(locations)})")

## 12. Model Comparison

We compare all models on the same validation set using average RMSE across all 83 counties.

Baseline results from `baseline_yuwenz.ipynb`:
- Persistence: 93.69
- Historical Median: 277.77
- Hurdle: 106563.83 (diverged)
- ZIP: 280.65

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2)))

def evaluate_avg_rmse(pred_df, truth_arr, locations):
    H = truth_arr.shape[0]
    rmses = []
    for i, loc in enumerate(locations):
        pv = pred_df[pred_df['location'].astype(str) == str(loc)]['pred'].values
        if len(pv) >= H:
            rmses.append(rmse(truth_arr[:, i], pv[:H]))
    return np.nanmean(rmses)

# Ground truth
val_truth_24h = ds_train_clean.out.transpose("timestamp", "location").isel(
    timestamp=slice(split_idx, split_idx + 24)).values.astype(float)
val_truth_48h = ds_train_clean.out.transpose("timestamp", "location").isel(
    timestamp=slice(split_idx, split_idx + 48)).values.astype(float)

# LSTM results
rmse_simple_24h = evaluate_avg_rmse(val_simple_24h, val_truth_24h, locations)
rmse_simple_48h = evaluate_avg_rmse(val_simple_48h, val_truth_48h, locations)
rmse_full_24h   = evaluate_avg_rmse(val_full_24h,   val_truth_24h, locations)
rmse_full_48h   = evaluate_avg_rmse(val_full_48h,   val_truth_48h, locations)

# Zero baseline
rmse_zero_24h = float(np.sqrt(np.mean(val_truth_24h ** 2)))
rmse_zero_48h = float(np.sqrt(np.mean(val_truth_48h ** 2)))

# Full comparison table including baselines from baseline_yuwenz.ipynb
results = pd.DataFrame({
    'Model': [
        'Zero Baseline',
        'Persistence',
        'Historical Median',
        'Hurdle',
        'ZIP',
        'LSTM-Simple (top 10 weather)',
        'LSTM-Full (all 77 weather)'
    ],
    'RMSE 24h': [
        round(rmse_zero_24h, 4),
        93.6870,
        277.7680,
        106563.8301,
        280.6458,
        round(rmse_simple_24h, 4),
        round(rmse_full_24h, 4)
    ],
    'RMSE 48h': [
        round(rmse_zero_48h, 4),
        'N/A',
        'N/A',
        'N/A',
        'N/A',
        round(rmse_simple_48h, 4),
        round(rmse_full_48h, 4)
    ]
})

print("=" * 65)
print("Model Comparison — Validation RMSE")
print("=" * 65)
print(results.to_string(index=False))

## 13. Visualization — Predictions vs Ground Truth

We plot predictions for the top 5 counties by mean outage count to visually compare LSTM-Simple and LSTM-Full against the ground truth.

In [ ]:
# Top 5 counties by mean outage
mean_out  = ds_train_clean.out.mean(dim="timestamp").values
top5_idx  = np.argsort(mean_out)[::-1][:5]
top5_locs = [locations[i] for i in top5_idx]

fig, axes = plt.subplots(5, 1, figsize=(15, 20))
fig.suptitle("Validation Predictions vs Ground Truth (24h) — Top 5 Counties",
             fontsize=14, fontweight='bold')

for ax, loc in zip(axes, top5_locs):
    loc_idx     = locations.index(loc)
    truth       = val_truth_24h[:, loc_idx]
    simple_pred = val_simple_24h[val_simple_24h['location'].astype(str) == str(loc)]['pred'].values[:24]
    full_pred   = val_full_24h[val_full_24h['location'].astype(str) == str(loc)]['pred'].values[:24]

    ax.plot(truth, label='Ground Truth', color='black', linewidth=2)
    if len(simple_pred) == 24:
        ax.plot(simple_pred, label='LSTM-Simple', color='steelblue', linewidth=1.5, linestyle='--')
    if len(full_pred) == 24:
        ax.plot(full_pred, label='LSTM-Full', color='darkorange', linewidth=1.5)

    ax.set_title(f"Location {loc}  (mean outage: {mean_out[loc_idx]:.1f})")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Outage Count")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "validation_predictions_24h.png"), dpi=120)
plt.show()

In [ ]:
# Bar chart comparison
plot_models  = ['Zero\nBaseline', 'Persistence', 'Hist\nMedian', 'Hurdle', 'ZIP',
                'LSTM\nSimple', 'LSTM\nFull']
plot_vals_24 = [rmse_zero_24h, 93.6870, 277.7680, 106563.8301, 280.6458,
                rmse_simple_24h, rmse_full_24h]

# Cap Hurdle for visualization
plot_vals_24_capped = [min(v, 500) for v in plot_vals_24]
colors = ['#d4e6f1'] * 5 + ['#2980b9', '#1a5276']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(plot_models, plot_vals_24_capped, color=colors, edgecolor='white')
for bar, val in zip(bars, plot_vals_24):
    label = f'{val:.1f}' if val < 500 else f'{val:.0f}\n(capped)'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            label, ha='center', va='bottom', fontsize=9)
ax.set_title("Model Comparison — Validation RMSE (24h)", fontsize=13, fontweight='bold')
ax.set_ylabel("Average RMSE")
ax.set_ylim(0, 550)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "model_comparison_bar.png"), dpi=120)
plt.show()

## 14. Generate Test Set Predictions

We retrain both LSTM models on the **full training data** (train + val) before generating test predictions. This gives the model as much data as possible before the final evaluation.

In [ ]:
# Retrain on full data
mu_simple_full, sd_simple_full = fit_scaler(X_simple)
mu_full_full,   sd_full_full   = fit_scaler(X_full)

X_simple_n = apply_scaler(np.nan_to_num(X_simple, 0), mu_simple_full, sd_simple_full)
X_full_n   = apply_scaler(np.nan_to_num(X_full, 0),   mu_full_full,   sd_full_full)

X_win_simple_24_final, y_win_24_final = build_windows(X_simple_n, y, SEQ_LEN, 24)
X_win_simple_48_final, y_win_48_final = build_windows(X_simple_n, y, SEQ_LEN, 48)
X_win_full_24_final, _                = build_windows(X_full_n,   y, SEQ_LEN, 24)
X_win_full_48_final, _                = build_windows(X_full_n,   y, SEQ_LEN, 48)

print("Retraining on full data...")
final_simple_24h, _ = train_model(X_win_simple_24_final, y_win_24_final, X_win_simple_24_final.shape[-1], 24, model_name="Final LSTM-Simple")
final_simple_48h, _ = train_model(X_win_simple_48_final, y_win_48_final, X_win_simple_48_final.shape[-1], 48, model_name="Final LSTM-Simple")
final_full_24h, _   = train_model(X_win_full_24_final,   y_win_24_final, X_win_full_24_final.shape[-1],   24, model_name="Final LSTM-Full")
final_full_48h, _   = train_model(X_win_full_48_final,   y_win_48_final, X_win_full_48_final.shape[-1],   48, model_name="Final LSTM-Full")
print("\n✓ Final models trained!")

In [ ]:
test_24h_ts = pd.to_datetime(ds_test_24h.timestamp.values)
test_48h_ts = pd.to_datetime(ds_test_48h.timestamp.values)

# Use LSTM-Full as the final submission (better RMSE)
test_pred_24h = predict_with_timestamps(final_full_24h, X_full_n, 24, locations, test_24h_ts)
test_pred_48h = predict_with_timestamps(final_full_48h, X_full_n, 48, locations, test_48h_ts)

test_pred_24h.to_csv(os.path.join(RESULTS_DIR, "lstm_pred_24h.csv"), index=False)
test_pred_48h.to_csv(os.path.join(RESULTS_DIR, "lstm_pred_48h.csv"), index=False)

print(f"✓ Saved lstm_pred_24h.csv — {len(test_pred_24h)} rows")
print(f"✓ Saved lstm_pred_48h.csv — {len(test_pred_48h)} rows")

## 15. Sanity Check

In [ ]:
n_loc = len(locations)
df_24 = pd.read_csv(os.path.join(RESULTS_DIR, "lstm_pred_24h.csv"))
df_48 = pd.read_csv(os.path.join(RESULTS_DIR, "lstm_pred_48h.csv"))

assert df_24.shape == (24 * n_loc, 3), f"Expected ({24*n_loc}, 3), got {df_24.shape}"
assert df_48.shape == (48 * n_loc, 3), f"Expected ({48*n_loc}, 3), got {df_48.shape}"
assert df_24.columns.tolist() == ['timestamp', 'location', 'pred']
assert df_48.columns.tolist() == ['timestamp', 'location', 'pred']
assert (df_24['pred'] >= 0).all(), "Negative predictions found"
assert (df_48['pred'] >= 0).all(), "Negative predictions found"

print("✓ All sanity checks passed!")
print(df_24.head())